In [6]:
# =============================================================================
# BLOCK 1: SETUP, IMPORTS, AND DATA LOADING
# =============================================================================
import warnings
warnings.filterwarnings('ignore')
import time
import os
# --- Library Imports ---
import pandas as pd
import numpy as np
import gc
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_squared_error
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import catboost as cb
import optuna
print("Libraries imported successfully.")
# --- Helper Function for Winkler Score ---
def winkler_score(y_true, lower, upper, alpha=0.1, return_coverage=False):
    width = upper - lower
    penalty_lower = np.where(y_true < lower, (2 / alpha) * (lower - y_true), 0)
    penalty_upper = np.where(y_true > upper, (2 / alpha) * (y_true - upper), 0)
    score = width + penalty_lower + penalty_upper
    if return_coverage:
        coverage = np.mean((y_true >= lower) & (y_true <= upper))
        return np.mean(score), coverage
    return np.mean(score)
# --- Global Constants ---
N_SPLITS = 5
RANDOM_STATE = 42
DATA_PATH = './'
N_OPTUNA_TRIALS = 30 # A strong number for a comprehensive search
COMPETITION_ALPHA = 0.1

# --- Load Raw Data ---
try:
    # We drop the low-variance columns they identified right away
    drop_cols=['id', 'golf', 'view_rainier', 'view_skyline', 'view_lakesamm','view_otherwater', 'view_other']
    df_train = pd.read_csv(DATA_PATH + 'dataset.csv').drop(columns=drop_cols)
    df_test = pd.read_csv(DATA_PATH + 'test.csv').drop(columns=drop_cols)
    print("Raw data loaded successfully.")
except FileNotFoundError:
    print("ERROR: Could not find 'dataset.csv' or 'test.csv'.")
    exit()
# --- Prepare Target Variable ---
y_true = df_train['sale_price'].copy()
# The mean-error model works best when predicting the raw price directly
# So, we will NOT log-transform the target this time.
# df_train.drop('sale_price', axis=1, inplace=True) # We keep sale_price for FE
print("Setup complete.")


Libraries imported successfully.
Raw data loaded successfully.
Setup complete.


In [7]:
# Make sure to have these libraries installed
# pip install pandas numpy scikit-learn

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
import gc

# Define a random state for reproducibility
RANDOM_STATE = 42

def create_comprehensive_features(df_train, df_test):
    """
    Combines original and new advanced feature engineering steps into a single pipeline.
    """
    print("--- Starting Comprehensive Feature Engineering ---")

    # Store original indices and target variable
    train_ids = df_train.index
    test_ids = df_test.index
    y_train = df_train['sale_price'].copy() # Keep the target separate

    # Combine for consistent processing
    df_train_temp = df_train.drop(columns=['sale_price'])
    all_data = pd.concat([df_train_temp, df_test], axis=0, ignore_index=True)

    # --- Original Feature Engineering ---

    # A) Brute-Force Numerical Interactions
    print("Step 1: Creating brute-force numerical interaction features...")
    NUMS = ['area', 'land_val', 'imp_val', 'sqft_lot', 'sqft', 'sqft_1', 'grade', 'year_built']
    # Ensure all columns exist and are numeric, fill missing with 0 for safety
    for col in NUMS:
        if col not in all_data.columns:
            all_data[col] = 0
        else:
            all_data[col] = pd.to_numeric(all_data[col], errors='coerce').fillna(0)
            
    for i in range(len(NUMS)):
        for j in range(i + 1, len(NUMS)):
            all_data[f'{NUMS[i]}_x_{NUMS[j]}'] = all_data[NUMS[i]] * all_data[NUMS[j]]

    # B) Date Features
    print("Step 2: Creating date features...")
    all_data['sale_date'] = pd.to_datetime(all_data['sale_date'])
    all_data['sale_year'] = all_data['sale_date'].dt.year
    all_data['sale_month'] = all_data['sale_date'].dt.month
    all_data['sale_dayofyear'] = all_data['sale_date'].dt.dayofyear
    all_data['age_at_sale'] = all_data['sale_year'] - all_data['year_built']

    # C) TF-IDF Text Features
    print("Step 3: Creating TF-IDF features for text columns...")
    text_cols = ['subdivision', 'zoning', 'city', 'sale_warning', 'join_status', 'submarket']
    all_data[text_cols] = all_data[text_cols].fillna('missing').astype(str)
    
    for col in text_cols:
        tfidf = TfidfVectorizer(analyzer='char', ngram_range=(3, 5), max_features=128, binary=True)
        svd = TruncatedSVD(n_components=8, random_state=RANDOM_STATE)
        
        tfidf_matrix = tfidf.fit_transform(all_data[col])
        tfidf_svd = svd.fit_transform(tfidf_matrix)
        
        tfidf_df = pd.DataFrame(tfidf_svd, columns=[f'{col}_tfidf_svd_{i}' for i in range(8)])
        all_data = pd.concat([all_data, tfidf_df], axis=1)

    # D) Log transform some interaction features
    for c in ['land_val_x_imp_val', 'land_val_x_sqft', 'imp_val_x_sqft']:
        if c in all_data.columns:
            all_data[c] = np.log1p(all_data[c].fillna(0))

    # --- New Feature Engineering Ideas ---

    # F) Group-By Aggregation Features
    print("Step 4: Creating group-by aggregation features...")
    group_cols = ['submarket', 'city', 'zoning']
    num_cols_for_agg = ['grade', 'sqft', 'imp_val', 'land_val', 'age_at_sale']

    for group_col in group_cols:
        for num_col in num_cols_for_agg:
            agg_stats = all_data.groupby(group_col)[num_col].agg(['mean', 'std', 'max', 'min']).reset_index()
            agg_stats.columns = [group_col] + [f'{group_col}_{num_col}_{stat}' for stat in ['mean', 'std', 'max', 'min']]
            all_data = pd.merge(all_data, agg_stats, on=group_col, how='left')
            all_data[f'{num_col}_minus_{group_col}_mean'] = all_data[num_col] - all_data[f'{group_col}_{num_col}_mean']

    # G) Ratio Features
    print("Step 5: Creating ratio features...")
    # Add a small epsilon to prevent division by zero
    epsilon = 1e-6 
    all_data['total_val'] = all_data['imp_val'] + all_data['land_val']
    all_data['imp_val_to_land_val_ratio'] = all_data['imp_val'] / (all_data['land_val'] + epsilon)
    all_data['land_val_ratio'] = all_data['land_val'] / (all_data['total_val'] + epsilon)
    all_data['sqft_to_lot_ratio'] = all_data['sqft'] / (all_data['sqft_lot'] + epsilon)
    all_data['was_renovated'] = (all_data['year_reno'] > 0).astype(int)
    all_data['reno_age_at_sale'] = np.where(all_data['was_renovated'] == 1, all_data['sale_year'] - all_data['year_reno'], -1)

    # H) Geospatial Clustering Features
    print("Step 6: Creating geospatial clustering features...")
    coords = all_data[['latitude', 'longitude']].copy()
    coords.fillna(coords.median(), inplace=True) # Simple imputation

    # KMeans is sensitive to feature scaling, but for lat/lon it's often okay without it.
    kmeans = KMeans(n_clusters=20, random_state=RANDOM_STATE, n_init=10) 
    all_data['location_cluster'] = kmeans.fit_predict(coords)
    
    # Calculate distance to each cluster center
    cluster_centers = kmeans.cluster_centers_
    for i in range(len(cluster_centers)):
        center = cluster_centers[i]
        all_data[f'dist_to_cluster_{i}'] = np.sqrt((coords['latitude'] - center[0])**2 + (coords['longitude'] - center[1])**2)


    # --- Final Cleanup ---
    print("Step 7: Finalizing feature set...")
    cols_to_drop = ['sale_date', 'subdivision', 'zoning', 'city', 'sale_warning', 'join_status', 'submarket']
    all_data = all_data.drop(columns=cols_to_drop)
    
    # One-hot encode the new cluster feature
    all_data = pd.get_dummies(all_data, columns=['location_cluster'], prefix='loc_cluster')
    
    # Final check for any remaining object columns
    object_cols = all_data.select_dtypes(include='object').columns
    if len(object_cols) > 0:
        all_data = all_data.drop(columns=object_cols)
        
    all_data.fillna(0, inplace=True)
    
    # === THE CRUCIAL FIX IS HERE ===
    # Convert ALL columns to a consistent numerical type to prevent errors.
    # float32 is memory-efficient and perfect for ML models.
    all_data = all_data.astype(np.float32)
    print("All feature columns successfully converted to float32.")


    # Separate back into train and test sets
    train_len = len(train_ids)
    X = all_data.iloc[:train_len].copy()
    X_test = all_data.iloc[train_len:].copy()
    
    # Restore original indices
    X.index = train_ids
    X_test.index = test_ids
    
    # Align columns - crucial for model prediction
    X_test = X_test[X.columns]
    
    print(f"\nComprehensive FE complete. Total features: {X.shape[1]}")
    gc.collect()
    
    return X, X_test, y_train
# =============================================================================
# BLOCK 2.5: EXECUTE FEATURE ENGINEERING
# =============================================================================
print("\n--- Starting Block 2.5: Executing Feature Engineering Pipeline ---")

# This is the crucial step that was missing.
# We call the function to create our training and testing dataframes.
X, X_test, y_train = create_comprehensive_features(df_train, df_test)

# Let's verify the output
print(f"Feature engineering complete. X shape: {X.shape}, X_test shape: {X_test.shape}")
gc.collect()

# =============================================================================
# BLOCK 2.6: ROBUST FEATURE CLIPPING (THE FINAL FIX)
# =============================================================================
print("\n--- Starting Block 2.6: Applying Robust Feature Clipping ---")
print("This step will protect the model from extreme outlier values in the test set.")

# We will iterate through each feature column
for col in X.columns:
    # Calculate the lower and upper bounds based ONLY on the training data distribution
    # Using the 0.1th and 99.9th percentiles is a robust way to handle outliers.
    lower_bound = X[col].quantile(0.001)
    upper_bound = X[col].quantile(0.999)
    
    # Count how many values in the test set are outside these bounds
    test_outliers = X_test[(X_test[col] < lower_bound) | (X_test[col] > upper_bound)][col].count()
    
    if test_outliers > 0:
        print(f"  - Feature '{col}': Found and clipped {test_outliers} extreme outliers.")
        
    # Apply the clipping to both the train and test sets
    # This ensures no value in the test set is outside the range seen in training.
    X[col] = X[col].clip(lower_bound, upper_bound)
    X_test[col] = X_test[col].clip(lower_bound, upper_bound)

print("\nRobust feature clipping complete. The data is now stabilized for scaling and training.")
gc.collect()


--- Starting Block 2.5: Executing Feature Engineering Pipeline ---
--- Starting Comprehensive Feature Engineering ---
Step 1: Creating brute-force numerical interaction features...
Step 2: Creating date features...
Step 3: Creating TF-IDF features for text columns...
Step 4: Creating group-by aggregation features...
Step 5: Creating ratio features...
Step 6: Creating geospatial clustering features...
Step 7: Finalizing feature set...
All feature columns successfully converted to float32.

Comprehensive FE complete. Total features: 233
Feature engineering complete. X shape: (200000, 233), X_test shape: (200000, 233)

--- Starting Block 2.6: Applying Robust Feature Clipping ---
This step will protect the model from extreme outlier values in the test set.
  - Feature 'sale_nbr': Found and clipped 113 extreme outliers.
  - Feature 'latitude': Found and clipped 384 extreme outliers.
  - Feature 'longitude': Found and clipped 400 extreme outliers.
  - Feature 'land_val': Found and clipped 1

0

In [8]:
# =============================================================================
# BLOCK: TUNE, TRAIN, & SAVE CATBOOST MEAN MODEL
# =============================================================================
import catboost as cb
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import mean_squared_error
import optuna
import numpy as np
import pandas as pd
import os
import gc

# --- 0. Define Constants ---
# Making this block self-contained.
N_OPTUNA_TRIALS = 30  # Number of tuning trials to run
N_SPLITS = 5          # Number of folds for cross-validation
RANDOM_STATE = 42     # Ensures reproducibility

# --- 1. Prepare Data for Tuning ---
print("--- Step 1: Preparing data for faster Optuna tuning... ---")
# This step requires that 'X' and 'y_true' have been created by the feature engineering block.
X_train_opt, X_val_opt, y_train_opt, y_val_opt = train_test_split(
    X, y_true, test_size=0.2, random_state=RANDOM_STATE
)
print(f"Data prepared. Training set size: {len(X_train_opt)}, Validation set size: {len(X_val_opt)}")

# --- 2. Define the Optuna Objective Function for CatBoost ---
def objective_catboost(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 1000, 3500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth': trial.suggest_int('depth', 6, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-2, 10.0, log=True),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'random_strength': trial.suggest_float('random_strength', 1e-2, 10.0, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'loss_function': 'RMSE', 'eval_metric': 'RMSE', 'random_seed': RANDOM_STATE, 'verbose': 0
    }
    model = cb.CatBoostRegressor(**params)
    model.fit(X_train_opt, y_train_opt, eval_set=[(X_val_opt, y_val_opt)], early_stopping_rounds=100, use_best_model=True)
    preds = model.predict(X_val_opt)
    rmse = np.sqrt(mean_squared_error(y_val_opt, preds))
    print(f"  Trial {trial.number}: RMSE = ${rmse:,.2f}")
    return rmse

# --- 3. Run the Optuna Study ---
study_catboost = optuna.create_study(direction='minimize')
print("\n--- Step 2: Starting CatBoost Hyperparameter Tuning... ---")
study_catboost.optimize(objective_catboost, n_trials=N_OPTUNA_TRIALS)
best_params_cb = study_catboost.best_params
print("\n--- Step 3: CatBoost Tuning Complete ---")
print(f"Best trial validation RMSE: ${study_catboost.best_value:,.2f}")
print("Best hyperparameters found:", best_params_cb)

# --- 4. K-Fold Cross-Validation and Prediction Generation ---
print("\n" + "="*80)
print("--- Step 4: K-Fold Cross-Validation with Optimal Hyperparameters ---")
print("="*80)
print("This step trains 5 separate CatBoost models to generate robust out-of-fold (OOF) predictions.")

# Initialize arrays to store the predictions
oof_catboost_preds = np.zeros(len(X))
test_catboost_preds = np.zeros(len(X_test))

# Use StratifiedKFold to ensure folds are balanced
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
for fold, (train_idx, val_idx) in enumerate(skf.split(X, grade_for_stratify)):
    print(f"\n--- Training Fold {fold+1}/{N_SPLITS} ---")
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train_fold, y_val_fold = y_true.iloc[train_idx], y_true.iloc[val_idx]

    # Initialize and train the model for this fold using the best parameters
    model = cb.CatBoostRegressor(**best_params_cb, verbose=0)
    model.fit(X_train, y_train_fold, eval_set=[(X_val, y_val_fold)], early_stopping_rounds=100, use_best_model=True)
    
    # Generate predictions for the validation set (this fold's OOF part)
    oof_preds_fold = model.predict(X_val)
    oof_catboost_preds[val_idx] = oof_preds_fold
    
    # Generate predictions for the test set and accumulate them
    test_catboost_preds += model.predict(X_test) / N_SPLITS
    
    fold_rmse = np.sqrt(mean_squared_error(y_val_fold, oof_preds_fold))
    print(f"  Fold {fold+1} Validation RMSE: ${fold_rmse:,.2f}")
    del model, X_train, X_val, y_train_fold, y_val_fold
    gc.collect()

# --- 5. Final Evaluation and Saving ---
print("\n" + "="*80)
print("--- Step 5: Final Evaluation and Saving Predictions ---")
print("="*80)

# Calculate the overall OOF RMSE across all folds
final_oof_rmse = np.sqrt(mean_squared_error(y_true, oof_catboost_preds))
print(f"Final CatBoost OOF RMSE across all {N_SPLITS} folds: ${final_oof_rmse:,.2f}")
print("This is the most reliable metric for this model's performance.")

# Define the save path and create the directory if it doesn't exist
SAVE_PATH = './kfold_oof_predictions/'
os.makedirs(SAVE_PATH, exist_ok=True)
print(f"\nPrediction arrays will be saved in: '{SAVE_PATH}'")

# Save the final OOF and Test prediction arrays, replacing any old files
np.save(os.path.join(SAVE_PATH, 'oof_catboost_preds.npy'), oof_catboost_preds)
np.save(os.path.join(SAVE_PATH, 'test_catboost_preds.npy'), test_catboost_preds)
print("\n'oof_catboost_preds.npy' and 'test_catboost_preds.npy' saved successfully.")

--- Step 1: Preparing data for faster Optuna tuning... ---


[I 2025-07-22 23:55:49,682] A new study created in memory with name: no-name-22246306-184b-4cf2-a858-f11d64f1d519


Data prepared. Training set size: 160000, Validation set size: 40000

--- Step 2: Starting CatBoost Hyperparameter Tuning... ---


[I 2025-07-22 23:57:20,668] Trial 0 finished with value: 102041.003472404 and parameters: {'iterations': 1705, 'learning_rate': 0.012776118635480881, 'depth': 10, 'l2_leaf_reg': 0.11869352506962007, 'subsample': 0.7606191074863352, 'random_strength': 0.0844769482606226, 'bagging_temperature': 0.2832977789220067}. Best is trial 0 with value: 102041.003472404.


  Trial 0: RMSE = $102,041.00


...

[I 2025-07-23 00:19:52,419] Trial 28 finished with value: 98854.02060002652 and parameters: {'iterations': 2012, 'learning_rate': 0.059613907841643006, 'depth': 7, 'l2_leaf_reg': 3.1896178540027607, 'subsample': 0.9756710486786713, 'random_strength': 0.021562324265618295, 'bagging_temperature': 0.716520597300496}. Best is trial 19 with value: 96705.60452557907.


  Trial 28: RMSE = $98,854.02


[I 2025-07-23 00:22:06,591] Trial 29 finished with value: 97253.4350056122 and parameters: {'iterations': 2383, 'learning_rate': 0.04162945039395614, 'depth': 10, 'l2_leaf_reg': 0.758367001231802, 'subsample': 0.943558832012389, 'random_strength': 0.13496928934367444, 'bagging_temperature': 0.2976365360654191}. Best is trial 19 with value: 96705.60452557907.


  Trial 29: RMSE = $97,253.44

--- Step 3: CatBoost Tuning Complete ---
Best trial validation RMSE: $96,705.60
Best hyperparameters found: {'iterations': 3500, 'learning_rate': 0.06698896253828576, 'depth': 8, 'l2_leaf_reg': 4.967920669444636, 'subsample': 0.997669491814972, 'random_strength': 0.013017982193191705, 'bagging_temperature': 0.6147919497446277}

--- Step 4: K-Fold Cross-Validation with Optimal Hyperparameters ---
This step trains 5 separate CatBoost models to generate robust out-of-fold (OOF) predictions.


NameError: name 'grade_for_stratify' is not defined

In [9]:
# =============================================================================
# BLOCK: K-FOLD TRAINING & SAVING WITH OPTIMAL CATBOOST PARAMETERS
# =============================================================================
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd
import os
import gc

# --- Step 1: Define the Best Hyperparameters ---
# These are the exact results from your successful tuning process (Trial 19).
print("--- Step 1: Loading the best hyperparameters found by Optuna ---")
best_params_cb = {
    'iterations': 3500,
    'learning_rate': 0.06698896253828576,
    'depth': 8,
    'l2_leaf_reg': 4.967920669444636,
    'subsample': 0.997669491814972,
    'random_strength': 0.013017982193191705,
    'bagging_temperature': 0.6147919497446277,
    # Add fixed params for consistency
    'loss_function': 'RMSE',
    'eval_metric': 'RMSE',
    'random_seed': RANDOM_STATE
}
print("Best parameters loaded successfully.")

# --- Step 2: K-Fold Cross-Validation and Prediction Generation ---
print("\n" + "="*80)
print("--- Step 2: K-Fold Cross-Validation with Optimal Hyperparameters ---")
print("="*80)
print("This step trains 5 separate CatBoost models to generate robust out-of-fold (OOF) predictions.")

# Initialize arrays to store the predictions
oof_catboost_preds = np.zeros(len(X))
test_catboost_preds = np.zeros(len(X_test))

# === THE FIX IS HERE ===
# Create the 'grade_for_stratify' variable from the original df_train.
# This ensures that skf.split() has the data it needs to create balanced folds.
grade_for_stratify = df_train['grade'].copy()
print("\n'grade_for_stratify' created successfully.")

# Use StratifiedKFold to ensure folds are balanced
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
for fold, (train_idx, val_idx) in enumerate(skf.split(X, grade_for_stratify)):
    print(f"\n--- Training Fold {fold+1}/{N_SPLITS} ---")
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train_fold, y_val_fold = y_true.iloc[train_idx], y_true.iloc[val_idx]

    # Initialize and train the model for this fold using the best parameters
    model = cb.CatBoostRegressor(**best_params_cb, verbose=0)
    model.fit(X_train, y_train_fold, eval_set=[(X_val, y_val_fold)], early_stopping_rounds=100, use_best_model=True)
    
    # Generate predictions for the validation set (this fold's OOF part)
    oof_preds_fold = model.predict(X_val)
    oof_catboost_preds[val_idx] = oof_preds_fold
    
    # Generate predictions for the test set and accumulate them (we average at the end)
    test_catboost_preds += model.predict(X_test)
    
    fold_rmse = np.sqrt(mean_squared_error(y_val_fold, oof_preds_fold))
    print(f"  Fold {fold+1} Validation RMSE: ${fold_rmse:,.2f}")
    del model, X_train, X_val, y_train_fold, y_val_fold
    gc.collect()

# --- Step 3: Final Evaluation and Saving ---
print("\n" + "="*80)
print("--- Step 3: Final Evaluation and Saving Predictions ---")
print("="*80)

# Average the test predictions by dividing by the number of folds
test_catboost_preds /= N_SPLITS

# Calculate the overall OOF RMSE across all folds
final_oof_rmse = np.sqrt(mean_squared_error(y_true, oof_catboost_preds))
print(f"Final CatBoost OOF RMSE across all {N_SPLITS} folds: ${final_oof_rmse:,.2f}")
print("This is the most reliable metric for this model's performance.")

# Define the save path and create the directory if it doesn't exist
SAVE_PATH = './kfold_oof_predictions/'
os.makedirs(SAVE_PATH, exist_ok=True)
print(f"\nPrediction arrays will be saved in: '{SAVE_PATH}'")

# Save the final OOF and Test prediction arrays, replacing any old files
np.save(os.path.join(SAVE_PATH, 'oof_catboost_preds.npy'), oof_catboost_preds)
np.save(os.path.join(SAVE_PATH, 'test_catboost_preds.npy'), test_catboost_preds)
print("\n'oof_catboost_preds.npy' and 'test_catboost_preds.npy' saved successfully.")

--- Step 1: Loading the best hyperparameters found by Optuna ---
Best parameters loaded successfully.

--- Step 2: K-Fold Cross-Validation with Optimal Hyperparameters ---
This step trains 5 separate CatBoost models to generate robust out-of-fold (OOF) predictions.

'grade_for_stratify' created successfully.

--- Training Fold 1/5 ---
  Fold 1 Validation RMSE: $96,425.14

--- Training Fold 2/5 ---
  Fold 2 Validation RMSE: $96,171.24

--- Training Fold 3/5 ---
  Fold 3 Validation RMSE: $96,690.83

--- Training Fold 4/5 ---
  Fold 4 Validation RMSE: $96,621.14

--- Training Fold 5/5 ---
  Fold 5 Validation RMSE: $96,440.68

--- Step 3: Final Evaluation and Saving Predictions ---
Final CatBoost OOF RMSE across all 5 folds: $96,469.98
This is the most reliable metric for this model's performance.

Prediction arrays will be saved in: './kfold_oof_predictions/'

'oof_catboost_preds.npy' and 'test_catboost_preds.npy' saved successfully.
